# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [18]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
parquet_files


['../../05_src/data/prices/FLIC/FLIC_1995/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1995/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2014/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2014/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1993/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1993/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2020/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2020/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2018/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2018/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2011/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2011/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2001/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2001/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2008/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2008/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1988/part.0.parquet

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [65]:
# Write your code below.
import shutil
# 'TEMP_DATA=../../05_src/data/temp/'
temp = os.getenv('TEMP_DATA')
temp_parquet = os.path.join(temp, 'parquet_hw')
shutil.rmtree(temp_parquet, ignore_errors=True)
os.makedirs(temp_parquet, exist_ok=True)

for parquet_file in parquet_files[0:1]:
    file = dd.read_parquet(parquet_file)#.set_index('ticker')    
    dd_feat = file.assign(
        Close_lag_1 = lambda x: x['Close'].shift(1),
        Adj_close_lag_1 = lambda x: x['Adj Close'].shift(1),
        Returns = lambda x: x['Close'] / x['Close_lag_1'] - 1,
        hi_lo_range = lambda x: x['High'] - x['Low']
    )
    src = dd_feat['source'].compute().unique()
    tck = dd_feat['ticker'].compute().unique()
    yr = dd_feat['Year'].compute().unique()
    temp_path = os.path.join(temp_parquet, *src, *tck, str(*yr))
    dd_feat.to_parquet(temp_path, engine='pyarrow')

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [69]:
# Write your code below.
import pandas as pd

parquet_files_changes = glob(os.path.join(temp_parquet, '**/*.parquet'), recursive=True)
for parquet_file in parquet_files_changes:
    dd.read_parquet(parquet_file)
    df_dd = dd_feat.compute()
    df_dd['moving_average_10_days'] = df_dd['Returns'].rolling(10).mean()
df_dd


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_close_lag_1,Returns,hi_lo_range,moving_average_10_days
145016,1995-01-03,2.485597,2.485597,2.485597,2.485597,1.176049,0.0,FLIC.csv,FLIC,1995,NaN,NaN,NaN,0.000000,NaN
145017,1995-01-04,2.485597,2.485597,2.485597,2.485597,1.176049,0.0,FLIC.csv,FLIC,1995,2.485597,1.176049,0.000000,0.000000,NaN
145018,1995-01-05,2.485597,2.485597,2.485597,2.485597,1.176049,0.0,FLIC.csv,FLIC,1995,2.485597,1.176049,0.000000,0.000000,NaN
145019,1995-01-06,2.485597,2.485597,2.485597,2.485597,1.176049,0.0,FLIC.csv,FLIC,1995,2.485597,1.176049,0.000000,0.000000,NaN
145020,1995-01-09,2.485597,2.625515,2.485597,2.625515,1.242250,5800.0,FLIC.csv,FLIC,1995,2.485597,1.176049,0.056291,0.139918,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145137,1995-06-26,2.781893,2.781893,2.781893,2.781893,1.316240,0.0,FLIC.csv,FLIC,1995,2.781893,1.316240,0.000000,0.000000,-0.003977
145138,1995-06-27,2.781893,2.781893,2.781893,2.781893,1.316240,0.0,FLIC.csv,FLIC,1995,2.781893,1.316240,0.000000,0.000000,-0.003977
145139,1995-06-28,2.781893,2.781893,2.781893,2.781893,1.316240,0.0,FLIC.csv,FLIC,1995,2.781893,1.316240,0.000000,0.000000,-0.003977
145140,1995-06-29,2.781893,2.781893,2.781893,2.781893,1.316240,0.0,FLIC.csv,FLIC,1995,2.781893,1.316240,0.000000,0.000000,-0.003977


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.